# Lab 20: The Red Circle Pops

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 3, *The Attention Filter*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-20.ipynb)

**What you will do:** time how long it takes to find a red circle among distractors that share its colour or its shape, turn your times into a cost per extra item, and compare your search slopes with the classic feature-integration study.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 20 you drew three search grids (Figure 3.1 in the book). When the red circle differed
from every distractor in colour alone, it "popped out" almost instantly, no matter how many
green circles surrounded it. When it shared its colour with some distractors and its shape
with others, you had to check items one by one, and the search got slower as you added more
items. The first is **parallel** (feature) search; the second is **serial** (conjunction)
search.

## Record your results

Repeat Steps 1-3 with a few different numbers of items, so you can see how much extra time
each additional item costs -- the same quantity Treisman and Gelade's slopes in
Figure 3.1c describe. Draw (or generate) grids of about 10, 20 and 30 items for the feature
search (Step 1 or 2) and for the conjunction search (Step 3), and time your friend on each.

In [ ]:
n_items = [10, 20, 30]  # distractors in each grid, matching Steps 1-3
feature_times_s = {10: 1.1, 20: 1.2, 30: 1.1}       # example data: replace with yours (Step 1/2)
conjunction_times_s = {10: 2.6, 20: 4.9, 30: 7.7}   # example data: replace with yours (Step 3)

times = pd.DataFrame({"feature search (s)": pd.Series(feature_times_s),
                       "conjunction search (s)": pd.Series(conjunction_times_s)})
times.index.name = "items in grid"
display(times)

(times * 1000).plot(marker="o", figsize=(6, 4), color=["steelblue", "firebrick"])
plt.xlabel("items in grid"); plt.ylabel("time to find the red circle (ms)")
plt.xticks(n_items); plt.show()

## Compare

Figure 3.1c reports the slopes from Treisman and Gelade's (1980) original study: feature
search costs about **3.1 ms per extra item**; conjunction search costs about **29 ms per
item** when the target is present, and about **67 ms per item** -- roughly twice as much --
when the reader has to decide it is absent. Fit a line through your own three points to get
your own slopes.

In [ ]:
feature_slope, _ = np.polyfit(n_items, times["feature search (s)"] * 1000, 1)
conjunction_slope, _ = np.polyfit(n_items, times["conjunction search (s)"] * 1000, 1)

print(f"Your feature-search slope:     {feature_slope:5.1f} ms/item  (Treisman & Gelade, 1980: 3.1 ms/item)")
print(f"Your conjunction-search slope: {conjunction_slope:5.1f} ms/item  (Treisman & Gelade, 1980: "
      "about 29 ms/item with the target present, about 67 ms/item when it is absent)")

if conjunction_slope > feature_slope:
    print("Your conjunction slope is the steeper of the two, as predicted by serial, item-by-item search.")
else:
    print("Your two slopes came out close together. A stopwatch and a single friend give a noisy estimate"
          " of a slope from only three points -- try more grid sizes, or check whether your tester scanned"
          " the whole grid at a glance rather than item by item.")

## The AI side

Feature-integration theory says the brain's early visual areas register colour, orientation
and other simple features in parallel, before attention binds them into objects. The early
layers of a convolutional neural network (CNN) trained on natural images learn a similar
set of feature detectors -- edges, oriented gratings, colour blobs -- without ever being
told what an edge is (Kriegeskorte, 2015). Below, we load a CNN pretrained on a million
photographs and look directly at its first-layer filters.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)
filters = model.conv1.weight.detach()  # shape: (64 filters, 3 colour channels, 7, 7 pixels)

def normalise(f):
    f = f - f.min()
    return (f / (f.max() + 1e-8)).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(8, 8, figsize=(10, 10))  # all 64 first-layer filters
for i, ax in enumerate(axes.flat):
    ax.imshow(normalise(filters[i]))
    ax.axis("off")
fig.suptitle("First-layer filters of a CNN trained on natural photographs (ResNet-18)")
plt.tight_layout(); plt.show()

print("Look for: oriented light/dark edges, colour-opponent blobs, and smooth colour gradients --")
print("the same feature types V1 neurons register before attention binds them into objects.")

These filters emerged purely from predicting image labels, with no instruction to build edge
or colour detectors -- much as the brain's preattentive feature maps are not "told" to
detect colour and orientation separately. But the analogy stops at the second stage: the CNN
has no separate "binding" step that takes time as the number of objects grows. It combines
all its filter responses for a whole image in one fixed-time forward pass, which is why a
computer search over a database (as the book's AI extension notes) does not show a
feature/conjunction slope difference at all.

## Pool the class data

Three grid sizes and one tester give a very noisy slope. Pool the class: each person adds
one row with an anonymous ID and their two slopes (in ms per item), downloads the sheet as
CSV, and loads it here. The rows below are placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,feature_slope_ms,conjunction_slope_ms
P01,1.0,25.0
P02,0.5,33.0
P03,1.5,19.0
P04,0.8,28.0
P05,1.2,31.0
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
boot = [df.conjunction_slope_ms.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean conjunction-search slope: {df.conjunction_slope_ms.mean():.1f} ms/item "
      f"(95% bootstrap interval {lo:.1f}-{hi:.1f} ms, n = {len(df)})")

df.plot.scatter(x="feature_slope_ms", y="conjunction_slope_ms", figsize=(5, 4), color="firebrick")
plt.xlabel("feature-search slope (ms/item)"); plt.ylabel("conjunction-search slope (ms/item)")
plt.axhline(29, color="grey", linestyle="--", label="Treisman & Gelade (1980) target-present: 29 ms/item")
plt.legend(); plt.show()

## Questions to think about

1. Which of your two slopes was steeper, and does that match Step 4's claim that a conjunction of features needs to be checked "item by item"?
2. The book reports that deciding a conjunction target is *absent* costs about twice as much per item (67 ms) as finding it when it is *present* (29 ms). Why would confirming an absence take about twice as long as finding a present target, if the search really does check items one at a time?
3. Look at the CNN filters in the AI Lab. Which filters look like colour blobs, and which look like oriented edges? Does the CNN keep colour and shape in separate "maps" the way feature-integration theory says the brain does at the preattentive stage?
4. Wolfe's Guided Search model (1994) says the visual system builds a single priority map from all the features at once, rather than checking a pure random order. If you ran Step 3 again on the same grid, would you expect your friend's time to change? What would that predict for a slope measured from repeated trials on the same display versus a fresh display each time?

## Challenge

Pop-out is not limited to colour. Design a new grid where every item is an identical grey
circle except one that is noticeably bigger (a size pop-out), and, if you have a phone or
tablet, one where every item is still except one that moves (a motion pop-out). Time your
friend on each and compare with your Step 1 colour pop-out time.

In [ ]:
# Example data: replace with your own timings, in seconds, for finding the odd item out
# when the pop-out feature is colour, size, or motion.
my_popout_times_s = {"colour (Step 1)": 0.9, "size": 1.0, "motion": 0.8}  # example data

results = pd.Series(my_popout_times_s, name="time (s)")
display(results.to_frame())
print("If size and motion pop out about as fast as colour, that supports the idea that several")
print("independent, low-level feature dimensions -- not colour specifically -- can drive")
print("parallel search (Wolfe, 1994).")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-20.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")